# Business Case #1 — Segmenting Bank Clients
**Politecnico di Milano — FinTech Lab**

Goal: segment 5,000 bank clients into meaningful Personas using Unsupervised ML (K-Means + alternatives).

---
## Pipeline
1. Load & Explore Data
2. Preprocessing (already done — reproduced here for completeness)
3. Find optimal k (Elbow + Silhouette)
4. Apply K-Means
5. Interpret Clusters → Personas
6. Visualizations
7. Alternative: Hierarchical Clustering
8. Assign new clients to Personas

---
## 1. Load & Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from scipy.cluster.hierarchy import dendrogram, linkage

# Load data
df = pd.read_excel('Dataset1_BankClients.xlsx')
df = df.drop(columns=['ID'])

print('--- Dataset Shape ---')
print(f'{df.shape[0]} clients, {df.shape[1]} features')

print('\n--- First 5 rows ---')
display(df.head())

print('\n--- Descriptive Statistics ---')
display(df.describe().round(2))

In [ ]:
# Quick visual overview of the dataset
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Dataset Overview', fontsize=16, fontweight='bold')

# Age distribution by gender
sns.histplot(data=df, x='Age', hue='Gender', multiple='stack', bins=20, ax=axes[0,0])
axes[0,0].set_title('Age Distribution by Gender')

# Job distribution
job_labels = {1:'Unemployed', 2:'Employee', 3:'Manager', 4:'Entrepreneur', 5:'Retired'}
df['Job_label'] = df['Job'].map(job_labels)
df['Job_label'].value_counts().plot(kind='bar', ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Job Distribution')
axes[0,1].tick_params(axis='x', rotation=30)

# Income vs Wealth
axes[0,2].scatter(df['Income'], df['Wealth'], alpha=0.1, color='darkorange')
axes[0,2].set_xlabel('Income')
axes[0,2].set_ylabel('Wealth')
axes[0,2].set_title('Income vs Wealth')

# Investments distribution
inv_labels = {1:'No investments', 2:'Lump sum', 3:'Capital accumulation'}
df['Inv_label'] = df['Investments'].map(inv_labels)
df['Inv_label'].value_counts().plot(kind='pie', ax=axes[1,0], autopct='%1.1f%%')
axes[1,0].set_title('Investment Type')
axes[1,0].set_ylabel('')

# Digital vs FinEdu
axes[1,1].scatter(df['Digital'], df['FinEdu'], alpha=0.1, color='green')
axes[1,1].set_xlabel('Digital Propensity')
axes[1,1].set_ylabel('Financial Education')
axes[1,1].set_title('Digital vs Financial Education')

# Area distribution
area_labels = {1:'Nord', 2:'Centro', 3:'Sud/Isole'}
df['Area_label'] = df['Area'].map(area_labels)
df['Area_label'].value_counts().plot(kind='bar', ax=axes[1,2], color='purple')
axes[1,2].set_title('Geographical Area')
axes[1,2].tick_params(axis='x', rotation=0)

# Drop helper columns
df = df.drop(columns=['Job_label', 'Inv_label', 'Area_label'])

plt.tight_layout()
plt.show()

---
## 2. Preprocessing

We treat the features in 3 groups:
- **Already scaled** (0–1 percentiles): Income, Wealth, Debt, FinEdu, ESG, Digital, BankFriend, LifeStyle, Luxury, Saving, Gender → taken as-is
- **Ordered categorical**: Age, CitySize, FamilySize → MinMaxScaler
- **Unordered categorical**: Job, Area, Investments → OneHotEncoder (drop='first')

In [ ]:
# --- Unordered categorical → One-Hot Encoding ---
no_order_columns = ['Job', 'Area', 'Investments']
no_order_features = df[no_order_columns].astype('category')

encoder = OneHotEncoder(drop='first', handle_unknown='ignore')
X_no_order = encoder.fit_transform(no_order_features).toarray()

# --- Ordered categorical → MinMaxScaler ---
order_columns = ['Age', 'CitySize', 'FamilySize']
order_features = df[order_columns]

scaler = MinMaxScaler()
X_order = scaler.fit_transform(order_features)

# --- Already scaled → take as-is ---
already_scaled_columns = ['Gender', 'Income', 'Wealth', 'Debt', 'FinEdu',
                           'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving']
X_num = df[already_scaled_columns].values

# --- Final matrix ---
X = np.hstack((X_num, X_order, X_no_order))

# Build named DataFrame for inspection
encoded_column_names = encoder.get_feature_names_out(no_order_columns).tolist()
all_columns = already_scaled_columns + order_columns + encoded_column_names
df_scaled = pd.DataFrame(X, columns=all_columns)

print(f'Matrix X shape: {X.shape}')  # should be (5000, 11+3+encoded)
display(df_scaled.head())

---
## 3. Find Optimal k — Elbow + Silhouette

In [ ]:
inertie = []
silhouette_scores = []
k_range = range(2, 11)

print('Computing KMeans for k = 2 to 10...')
for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X)
    inertie.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X, kmeans.labels_))
    print(f'  k={k} | Inertia: {kmeans.inertia_:.1f} | Silhouette: {silhouette_score(X, kmeans.labels_):.4f}')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal k Selection', fontsize=15, fontweight='bold')

ax1.plot(k_range, inertie, marker='o', color='steelblue', linewidth=2)
ax1.set_title('Elbow Method')
ax1.set_xlabel('Number of clusters k')
ax1.set_ylabel('Inertia')
ax1.grid(True, alpha=0.3)

best_k_sil = k_range[silhouette_scores.index(max(silhouette_scores))]
ax2.plot(k_range, silhouette_scores, marker='o', color='darkorange', linewidth=2)
ax2.axvline(x=best_k_sil, color='red', linestyle='--', label=f'Best k={best_k_sil}')
ax2.set_title('Silhouette Score')
ax2.set_xlabel('Number of clusters k')
ax2.set_ylabel('Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nBest k by Silhouette: {best_k_sil}')
print('→ Also inspect the Elbow plot and choose k where inertia drop slows down.')

---
## 4. Apply K-Means with chosen k

> **Change `K_CHOSEN` below based on the plots above.**

In [ ]:
# ← Change this value based on the Elbow + Silhouette plots
K_CHOSEN = best_k_sil

kmeans_final = KMeans(n_clusters=K_CHOSEN, n_init=20, random_state=42)
df['Cluster'] = kmeans_final.fit_predict(X)

# Cluster sizes
cluster_sizes = df['Cluster'].value_counts().sort_index()
print('--- Cluster Sizes ---')
for i, size in cluster_sizes.items():
    print(f'  Cluster {i}: {size} clients ({size/len(df)*100:.1f}%)')

---
## 5. Interpret Clusters → Personas

We analyze the **original (unscaled)** feature means per cluster to understand who each Persona is.

In [ ]:
# --- Centroid analysis on original data ---
cluster_profile = df.groupby('Cluster').mean().round(2)

print('--- Cluster Centroids (original values) ---')
display(cluster_profile)

# Map readable labels for categorical features
job_labels = {1:'Unemployed', 2:'Employee', 3:'Manager', 4:'Entrepreneur', 5:'Retired'}
area_labels = {1:'Nord', 2:'Centro', 3:'Sud/Isole'}
inv_labels  = {1:'No investments', 2:'Lump sum', 3:'Capital acc.'}

print('\n--- Key Features by Cluster ---')
summary = pd.DataFrame({
    'Cluster':      range(K_CHOSEN),
    'N clients':    [cluster_sizes[i] for i in range(K_CHOSEN)],
    'Avg Age':      cluster_profile['Age'].values,
    'Gender (F=1)': cluster_profile['Gender'].round(2).values,
    'Avg Job':      [job_labels[round(cluster_profile['Job'][i])] for i in range(K_CHOSEN)],
    'Income':       cluster_profile['Income'].values,
    'Wealth':       cluster_profile['Wealth'].values,
    'FinEdu':       cluster_profile['FinEdu'].values,
    'Digital':      cluster_profile['Digital'].values,
    'Investments':  [inv_labels[round(cluster_profile['Investments'][i])] for i in range(K_CHOSEN)]
})
display(summary)

In [ ]:
# --- Heatmap of cluster profiles ---
# Select most meaningful numerical features
features_to_plot = ['Age', 'Income', 'Wealth', 'Debt', 'FinEdu',
                    'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving',
                    'FamilySize', 'CitySize']

# Normalize cluster means to 0-1 for comparison
heatmap_data = cluster_profile[features_to_plot]
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

plt.figure(figsize=(14, 5))
sns.heatmap(heatmap_norm, annot=heatmap_data.round(2), fmt='g',
            cmap='RdYlGn', linewidths=0.5, cbar_kws={'label': 'Normalized value'})
plt.title('Cluster Profiles — Feature Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('Cluster')
plt.tight_layout()
plt.show()

---
## 6. Visualizations

In [ ]:
# --- Cluster size distribution ---
colors = plt.cm.tab10.colors

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Cluster Analysis', fontsize=16, fontweight='bold')

# Pie chart of cluster sizes
axes[0,0].pie(cluster_sizes.values, labels=[f'Cluster {i}' for i in cluster_sizes.index],
              autopct='%1.1f%%', colors=colors[:K_CHOSEN])
axes[0,0].set_title('Cluster Sizes')

# Age distribution by cluster
for i in range(K_CHOSEN):
    axes[0,1].hist(df[df['Cluster']==i]['Age'], bins=20, alpha=0.6,
                   label=f'Cluster {i}', color=colors[i])
axes[0,1].set_title('Age Distribution by Cluster')
axes[0,1].set_xlabel('Age')
axes[0,1].legend()

# Income vs Wealth colored by cluster
for i in range(K_CHOSEN):
    mask = df['Cluster'] == i
    axes[0,2].scatter(df[mask]['Income'], df[mask]['Wealth'],
                      alpha=0.2, color=colors[i], label=f'Cluster {i}', s=10)
axes[0,2].set_title('Income vs Wealth by Cluster')
axes[0,2].set_xlabel('Income')
axes[0,2].set_ylabel('Wealth')
axes[0,2].legend()

# Digital vs FinEdu
for i in range(K_CHOSEN):
    mask = df['Cluster'] == i
    axes[1,0].scatter(df[mask]['Digital'], df[mask]['FinEdu'],
                      alpha=0.2, color=colors[i], label=f'Cluster {i}', s=10)
axes[1,0].set_title('Digital vs Financial Education')
axes[1,0].set_xlabel('Digital Propensity')
axes[1,0].set_ylabel('Financial Education')
axes[1,0].legend()

# Gender by cluster
gender_by_cluster = df.groupby('Cluster')['Gender'].mean()
gender_by_cluster.plot(kind='bar', ax=axes[1,1], color=colors[:K_CHOSEN])
axes[1,1].set_title('Avg Gender by Cluster (1=Female)')
axes[1,1].set_ylabel('Proportion Female')
axes[1,1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
axes[1,1].tick_params(axis='x', rotation=0)

# Investment type by cluster
inv_by_cluster = df.groupby(['Cluster', 'Investments']).size().unstack(fill_value=0)
inv_by_cluster.columns = ['No inv.', 'Lump sum', 'Capital acc.']
inv_by_cluster.div(inv_by_cluster.sum(axis=1), axis=0).plot(
    kind='bar', ax=axes[1,2], stacked=True)
axes[1,2].set_title('Investment Type by Cluster')
axes[1,2].set_ylabel('Proportion')
axes[1,2].tick_params(axis='x', rotation=0)
axes[1,2].legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# --- Radar chart for cluster profiles ---
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

radar_features = ['Income', 'Wealth', 'FinEdu', 'Digital', 'ESG',
                  'Luxury', 'Saving', 'BankFriend']

# Normalize to 0-1
radar_data = cluster_profile[radar_features]
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min())

N = len(radar_features)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.suptitle('Cluster Radar Chart', fontsize=14, fontweight='bold')

for i in range(K_CHOSEN):
    values = radar_norm.iloc[i].tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, color=colors[i], label=f'Cluster {i}')
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_features, fontsize=11)
ax.set_ylim(0, 1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.show()

---
## 7. Alternative Approach — Hierarchical Clustering

We compare K-Means with Agglomerative Clustering (Ward method) to validate our segmentation.

In [ ]:
# Dendrogram on a sample (full dataset is too large to visualize)
np.random.seed(42)
sample_idx = np.random.choice(len(X), 200, replace=False)
X_sample = X[sample_idx]

linked = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(linked, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=8, show_contracted=True)
plt.title('Dendrogram — Hierarchical Clustering (sample of 200)', fontsize=13, fontweight='bold')
plt.xlabel('Client Index')
plt.ylabel('Distance (Ward)')
plt.tight_layout()
plt.show()

In [ ]:
# Apply Agglomerative Clustering with same k
agglo = AgglomerativeClustering(n_clusters=K_CHOSEN, linkage='ward')
df['Cluster_Agglo'] = agglo.fit_predict(X)

# Compare Silhouette scores
sil_kmeans = silhouette_score(X, df['Cluster'])
sil_agglo  = silhouette_score(X, df['Cluster_Agglo'])

print('--- Comparison: K-Means vs Hierarchical ---')
print(f'  K-Means      Silhouette: {sil_kmeans:.4f}')
print(f'  Hierarchical Silhouette: {sil_agglo:.4f}')
print(f'  → Better method: {"K-Means" if sil_kmeans >= sil_agglo else "Hierarchical"}')

# Cluster sizes comparison
print('\n--- Cluster sizes ---')
print('K-Means:     ', df['Cluster'].value_counts().sort_index().to_dict())
print('Hierarchical:', df['Cluster_Agglo'].value_counts().sort_index().to_dict())

---
## 8. Assign a New Client to a Persona

After training, any new client can be assigned to the closest Persona using the same preprocessing pipeline.

In [ ]:
# Example: new client to classify
new_client = pd.DataFrame([{
    'Age': 45,
    'Gender': 1,        # Female
    'Job': 3,           # Manager
    'Area': 1,          # Nord
    'CitySize': 3,      # Large city
    'FamilySize': 2,
    'Income': 0.75,
    'Wealth': 0.80,
    'Debt': 0.20,
    'FinEdu': 0.85,
    'ESG': 0.70,
    'Digital': 0.90,
    'BankFriend': 0.65,
    'LifeStyle': 0.75,
    'Luxury': 0.60,
    'Saving': 0.55,
    'Investments': 3    # Capital accumulation
}])

# Apply same preprocessing
X_new_no_order = encoder.transform(new_client[no_order_columns].astype('category')).toarray()
X_new_order    = scaler.transform(new_client[order_columns])
X_new_num      = new_client[already_scaled_columns].values
X_new          = np.hstack((X_new_num, X_new_order, X_new_no_order))

# Predict cluster
predicted_cluster = kmeans_final.predict(X_new)[0]

print(f'New client assigned to: Cluster {predicted_cluster}')
print(f'\nCluster {predicted_cluster} profile:')
display(cluster_profile.loc[predicted_cluster, ['Age','Income','Wealth','FinEdu','Digital','Investments']].to_frame().T)

---
## 9. Personas — Summary

> **Fill in the persona names and descriptions after analyzing the heatmap and cluster profiles above.**

| Cluster | Persona Name | Key Traits | Main Needs | Service Model |
|---------|-------------|-----------|------------|---------------|
| 0 | *(e.g. Young Digital)* | Young, high digital, low wealth | Savings, first investments | Digital |
| 1 | *(e.g. Wealthy Retiree)* | Older, high wealth, low digital | Capital protection, inheritance | Physical |
| ... | ... | ... | ... | ... |